# Gemma 4 리스크 분류 실험 (Kaggle)

파이프라인 `risk` 단계(Plan 03)가 근거 문장마다 붙이는 **뉘앙스·위험 유형·그 뜻으로 쓰였는지**를 Kaggle GPU 에서 **수집 서버와 같은 조건**
(같은 Ollama 버전·같은 모델 파일·같은 호출 옵션·같은 프롬프트)으로 다시 붙여 보고, 서버가 이미 붙인 분류·사람 판정과 비교한다.
프롬프트를 바꿔 가며 오탐(예: 문장 분위기가 부정적일 뿐인 `pov`·`brah`)을 줄이는 연구용이다.

> **⚠ 이 노트북에는 데이터가 없다. 수집 서버에서 표본을 받아 와야 실험이 된다.**
> 실데이터(검증 용어·근거 문장·서버가 붙인 분류)는 수집 서버의 DB 에만 있다. 개발 PC·Kaggle·저장소에는 없다.
> 아래 '준비' 2번대로 **서버에서 `export_risk_sample.py` 를 실행해 받은 두 파일**을 비공개 Kaggle Dataset 으로 올리고 이 노트북의 입력(**Add Input**)으로 붙인 뒤 실행한다.
> 파일 없이 실행하면 합성 문장 10개로만 돌아가며, 이때 나온 수치는 연구 결과로 쓰지 않는다.
> 스크립트는 새 코드(`apps/trend-api`, PostgreSQL)가 필요하다 — 서버가 옛 `prototype/trend` 로 돌고 있으면 먼저 `develop` 으로 옮긴다.

> 2026-10-01 에 Colab 용에서 Kaggle 용으로 옮겼다. **Kaggle 에서는 아직 한 번도 돌려 보지 않았다** — 첫 실행에서 합성 표본으로 설치·추론·저장·재개를 먼저 확인한다.

**그대로 옮긴 것** (`apps/trend-api` 기준): `pipeline/risk.py` 의 `PROMPT`·`PER_TERM`·판정 문턱, `schemas.EvidenceLabel`,
`interpreters/local_chat.py` 의 호출 방식(스키마는 프롬프트에, Ollama `format` 없음, `think=False`, `num_ctx` 8192, temperature 0.1, 틀리면 한 번 재질문, 맞는 항목만 살림).

**입력과 결과.** 첫 설정 셀이 `/kaggle/input` 아래에서 표본 CSV 한 개와 같은 이름의 `.env.json` 이 든 폴더를 찾는다(`INPUT_DIR` 에 직접 적어도 된다).
결과는 `/kaggle/working/runs/<RUN_ID>` 에 저장한다. `/kaggle/working` 은 **Save Version(Save & Run All)으로 돌린 버전의 Output 으로 남고**,
대화형 세션에서만 돌리면 세션이 끝날 때 사라진다. 남겨야 하는 실행은 Save Version 으로 돌린다(브라우저를 닫아도 끝까지 돈다).
이어서 돌릴 때는 이전 버전의 Output 을 입력으로 붙이고 `RESUME_FROM` 에 그 안의 `runs/<RUN_ID>` 폴더를 적는다 — 체크포인트를 새 실행 폴더로 복사해 끝난 용어는 건너뛴다.

**작은 실행 확인**
첫 설정 셀에서 `USE_SYNTHETIC_SAMPLE=True`로 지정하면 용어 1개·합성 문장 5개를 `/kaggle/working/inputs/<RUN_ID>` 에 만들고 입출력을 검증한다. GPU가 있으면 이후 셀에서 실제 Gemma 추론을 진행한다. GPU가 없으면 설치 전에 멈춘다. 이 모드는 품질 평가나 가중치 학습이 아니다. 사람 판정이 있는 실표본을 사용할 때는 `False`로 두고 표본 Dataset 을 입력으로 붙인다.

**준비**
1. Kaggle 계정의 **휴대폰 인증**을 마친다(인증 전에는 GPU·인터넷을 켤 수 없다). 노트북 오른쪽 **Settings** 에서 Accelerator 를 **GPU T4 x2** 로, **Internet** 을 켠다(Ollama 설치·모델 받기에 필요).
   무료 GPU 는 계정당 주간 할당(약 30시간)이 있고 한 세션은 최대 12시간이다. 남은 할당은 계정 설정에서 본다. 붐빌 때는 GPU 를 기다릴 수 있다. P100 에서는 확인하지 않았다.
2. 수집 서버에서 표본을 만든다. 문장은 `risk` 단계가 분류하는 것과 같다(커뮤니티·SNS 글 포함).
   ```bash
   cd apps/trend-api && uv run scripts/export_risk_sample.py --terms 40
   ```
   `artifacts/risk_sample/` 의 `risk_sample_*.csv` 와 같은 이름의 `.env.json`(서버의 Ollama 버전·모델 digest·옵션·프롬프트 해시) **두 파일을 함께** 비공개 Dataset 에 올린다.
   새 표본은 같은 Dataset 의 새 버전으로 올린다. `.env.json` 이 없으면 기본값으로 돌고 '같은 환경' 확인을 건너뛴다. 아무것도 붙이지 않으면 합성 문장으로 동작만 확인한다(수치 의미 없음).

**서버와 달라지는 것**: GPU 종류(속도, 아주 작은 수치 차이). 그래서 시간은 서버 값과 섞지 않는다. 문장 묶음도 다르다 — 서버는 아직 분류하지 않은
문장만 묶어 보내고, 여기서는 용어마다 최근 12개를 한 번에 묶는다.

**지킬 것**
- 올리는 것은 표본 CSV 와 `.env.json` 뿐이다. 저장소의 `.env`·키·사용자 입력 문장·DB 파일은 올리지 않는다.
- 표본에는 공개 기사·쇼츠·커뮤니티 글의 발췌가 들어 있고, 이 문장들이 Kaggle 에 올라간다. Dataset 과 노트북은 **Private** 으로 두고 필요한 팀원만 Collaborator 로 더한다. 공개로 바꾸지 않는다.
- 결과 파일에는 문장이 들어 있다. 저장소에 커밋하지 않는다.
- 모델 파일은 `/kaggle/working` 에 두지 않는다(Output 20GB 한도, 버전마다 저장됨). Ollama 는 기본 위치 `~/.ollama` 에 받는다.

**GitHub 과 코드 맞추기** — 편집기의 **File → Import Notebook** 으로 이 파일을 가져온다. 가져온 노트북은 Kaggle 쪽 사본이라 저장소가 바뀌어도 따라오지 않는다.
공통 코드를 고쳤으면 **File → Download Notebook** 으로 받아 개발 PC 에서 출력을 지우고(출력에 문장이 남는다) 작업 브랜치·PR 로 `develop` 에 반영한다.
Kaggle 에서 `develop`·`release/gcp` 로 바로 저장하지 않는다. 두 사람이 같은 파일을 동시에 고치지 않는다.


In [ ]:
# .env.json 을 올리면 MODEL·옵션·Ollama 버전은 서버 값으로 덮인다. 모델 크기 비교처럼 일부러 바꿀 때만 OVERRIDE 에 적는다.
OVERRIDE = {}               # 예: {"model": "gemma4:e4b"}
MODEL = "gemma4:12b"        # config.GEMMA_LOCAL_MODEL
TEMPERATURE = 0.1           # pipeline/local_llm.structured 기본값
NUM_CTX = 8192              # interpreters/local_chat.NUM_CTX
MAX_TOKENS = 2048           # interpreters/local_chat.MAX_TOKENS
PER_TERM = 12               # pipeline/risk.PER_TERM (문장은 내보낼 때 이미 이만큼으로 잘려 있다)
VARIANTS = ["baseline"]     # 아래 PROMPTS 의 이름. 비교하려면 예: ["baseline", "v2_term_vs_tone"]
MAX_TERMS = None            # 시험 삼아 몇 개만 돌리려면 숫자
USE_SYNTHETIC_SAMPLE = False  # True creates a five-sentence smoke sample in /kaggle/working
INPUT_DIR = ""              # 비우면 /kaggle/input 아래에서 risk_sample_*.csv 가 든 폴더를 찾는다
RESUME_FROM = ""            # 이어서 돌릴 때: 입력으로 붙인 이전 버전 Output 의 runs/<RUN_ID> 폴더
if USE_SYNTHETIC_SAMPLE:
    MAX_TERMS = 1
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import shutil

WORK_DIR = Path("/kaggle/working")
if not WORK_DIR.is_dir():
    raise RuntimeError("Kaggle working directory is unavailable")
if RESUME_FROM:
    if not (Path(RESUME_FROM) / "signature.json").is_file():
        raise FileNotFoundError(Path(RESUME_FROM) / "signature.json")
    RUN_ID = Path(RESUME_FROM).name
else:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
RUN_DIR = str(WORK_DIR / "runs" / RUN_ID)
if RESUME_FROM and not Path(RUN_DIR).exists():
    shutil.copytree(RESUME_FROM, RUN_DIR)
if USE_SYNTHETIC_SAMPLE:
    INPUT_DIR = str(WORK_DIR / "inputs" / RUN_ID)
elif not INPUT_DIR:
    found = sorted({p.parent for p in Path("/kaggle/input").rglob("risk_sample_*.csv")})
    if len(found) > 1:
        raise ValueError("표본 폴더가 여럿이다 — INPUT_DIR 에 하나를 적는다: " + ", ".join(map(str, found)))
    INPUT_DIR = str(found[0]) if found else ""
print("Input", INPUT_DIR or "(none, synthetic sample)")
print("Run", RUN_DIR)

## 1. 표본과 서버 환경 불러오기

In [ ]:
import io, json
import pandas as pd

SYNTHETIC = False
if USE_SYNTHETIC_SAMPLE or not INPUT_DIR:
    up = {}
else:
    inputs = sorted(Path(INPUT_DIR).glob("risk_sample_*.csv"))
    if len(inputs) != 1:
        raise ValueError("INPUT_DIR에는 표본 CSV가 정확히 하나 있어야 합니다")
    csv_path = inputs[0]
    env_path = csv_path.with_suffix(".env.json")
    if not env_path.is_file():
        raise FileNotFoundError(env_path)
    up = {csv_path.name: csv_path.read_bytes(), env_path.name: env_path.read_bytes()}

SERVER_ENV = {}
csvs = [k for k in up if k.endswith(".csv")]
envs = [k for k in up if k.endswith(".json")]
if envs:
    SERVER_ENV = json.loads(up[envs[0]].decode("utf-8"))
    SYNTHETIC = bool(SERVER_ENV.get("synthetic", False))
if csvs:
    SOURCE = csvs[0]
    df = pd.read_csv(io.BytesIO(up[SOURCE]), encoding="utf-8-sig", dtype=str).fillna("")
else:
    SYNTHETIC, SOURCE = True, "synthetic"
    print("⚠ 서버 표본을 입력으로 붙이지 않았다 — 합성 문장으로 동작만 확인한다. 수치는 연구 결과로 쓰지 않는다.")
    print("  수집 서버에서: cd apps/trend-api && uv run scripts/export_risk_sample.py --terms 40")
    rows = []
    fake = {
        "pov": ("어떤 상황을 1인칭 시점으로 연출하는 영상 형식", [
            "pov: 월요일 아침 알람이 세 번째 울릴 때", "pov 영상 찍는데 친구가 계속 웃음 참기 실패",
            "이 pov 너무 공감돼서 저장함", "pov 형식으로 브이로그 올렸어요", "요즘 pov 영상이 너무 많아서 좀 질림"]),
        "La Peace": ("", [
            "정부가 발표한 La Peace 정책에 대한 의회 표결이 연기됐다", "La Peace 협상 대표단이 다시 만났다",
            "La Peace 계획 반대 시위가 열렸다", "La Peace 합의문 초안이 공개됐다", "대통령은 La Peace 가 성과라고 말했다"]),
    }
    for term, (definition, sents) in fake.items():
        for i, s in enumerate(sents):
            rows.append({"term_id": f"syn_{term}", "term": term, "definition": definition, "term_risk_level": "",
                         "term_risk_human": "", "evidence_id": f"syn_{term}_{i}", "i": str(i), "kind": "news",
                         "source_domain": f"example{i % 2}.com", "community": "0", "observed_at": "", "sentence": s,
                         "pipeline_polarity": "", "pipeline_risk": "", "human_on_topic": "", "human_polarity": "",
                         "human_risk": "", "human_level": ""})
    df = pd.DataFrame(rows)

import hashlib
SOURCE_SHA256 = hashlib.sha256(up[SOURCE] if csvs else df.to_csv(index=False).encode()).hexdigest()
df["i"] = df["i"].astype(int)
terms = list(dict.fromkeys(df["term_id"]))[: MAX_TERMS or None]
print(f"{SOURCE}: 용어 {len(terms)}개 · 문장 {len(df[df.term_id.isin(terms)])}개 · 커뮤니티 문장 {int((df.community == '1').sum())}개")

# 서버 조건으로 맞춘다
if SERVER_ENV:
    if SERVER_ENV.get("backend") not in (None, "ollama"):
        print("⚠ 서버는 Ollama 가 아닌 OpenAI 호환 서버(vLLM 등)로 돌았다 — 여기 결과와 조건이 다르다")
    MODEL = SERVER_ENV.get("model", MODEL)
    TEMPERATURE = SERVER_ENV.get("temperature", TEMPERATURE)
    NUM_CTX = SERVER_ENV.get("num_ctx", NUM_CTX)
    MAX_TOKENS = SERVER_ENV.get("max_tokens", MAX_TOKENS)
    PER_TERM = SERVER_ENV.get("per_term", PER_TERM)
else:
    print("⚠ .env.json 이 없다 — 기본값으로 돌고 '같은 환경' 확인을 건너뛴다")
MODEL = OVERRIDE.get("model", MODEL)
OLLAMA_PIN = "" if "model" in OVERRIDE else SERVER_ENV.get("ollama_version", "")
print(f"model={MODEL} temperature={TEMPERATURE} num_ctx={NUM_CTX} max_tokens={MAX_TOKENS} per_term={PER_TERM} ollama={OLLAMA_PIN or 'latest'}")

if USE_SYNTHETIC_SAMPLE:
    df = df[df.term_id.isin(terms)].copy()
    input_path = Path(INPUT_DIR)
    input_path.mkdir(parents=True, exist_ok=True)
    sample_path = input_path / "risk_sample_smoke.csv"
    df.to_csv(sample_path, index=False, encoding="utf-8-sig")
    sample_env = {"synthetic": True, "backend": "ollama", "model": MODEL,
                  "temperature": TEMPERATURE, "num_ctx": NUM_CTX,
                  "max_tokens": MAX_TOKENS, "per_term": PER_TERM}
    sample_path.with_suffix(".env.json").write_text(json.dumps(sample_env, ensure_ascii=False, indent=2), encoding="utf-8")
    SOURCE = sample_path.name
    SOURCE_SHA256 = hashlib.sha256(sample_path.read_bytes()).hexdigest()

if RUN_DIR:
    import shutil, subprocess
    run_path = Path(RUN_DIR)
    run_path.mkdir(parents=True, exist_ok=True)
    gpu = ""
    if shutil.which("nvidia-smi"):
        probe = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
        if probe.returncode == 0:
            gpu = probe.stdout.strip()
    path_check = {"stage": "input_output_check", "synthetic": SYNTHETIC,
                  "source_sha256": SOURCE_SHA256, "terms": len(terms),
                  "sentences": len(df[df.term_id.isin(terms)]),
                  "input_read_verified": True, "output_write_read_verified": True,
                  "gpu_available": bool(gpu), "gpu": gpu, "inference_executed": False}
    check_path = run_path / "path_check.json"
    check_path.write_text(json.dumps(path_check, ensure_ascii=False, indent=2), encoding="utf-8")
    if json.loads(check_path.read_text(encoding="utf-8")) != path_check:
        raise RuntimeError("Output readback did not match")
    print("INPUT_OUTPUT_CHECK PASSED", check_path)
    print("GPU available", bool(gpu))

## 2. Ollama 설치와 모델 받기 (서버와 같은 버전)

In [ ]:
import shutil, subprocess
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise RuntimeError("GPU가 없습니다. Settings 에서 Accelerator 를 GPU T4 x2 로, Internet 을 켠 뒤 실행하세요")

!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_PIN} sh
!pip -q install ollama

In [ ]:
import json, os, subprocess, time, urllib.request

log = open("/tmp/ollama.log", "w")
server = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2) as r:
            OLLAMA_VERSION = json.load(r)["version"]
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("ollama serve 가 뜨지 않았다 — /tmp/ollama.log 를 본다")
print("ollama", OLLAMA_VERSION, "(서버와 같음)" if OLLAMA_VERSION == OLLAMA_PIN else f"⚠ 서버는 {OLLAMA_PIN or '?'}")

In [ ]:
!ollama pull {MODEL}
with urllib.request.urlopen("http://127.0.0.1:11434/api/tags") as r:
    MODEL_DIGEST = next((m["digest"] for m in json.load(r)["models"] if m["model"] == MODEL), "")
want = SERVER_ENV.get("model_digest", "") if "model" not in OVERRIDE else ""
print("digest", MODEL_DIGEST[:12], "(서버와 같음)" if want and MODEL_DIGEST == want else
      f"⚠ 서버는 {want[:12]} — 태그가 갱신됐을 수 있다. 서버에서 `ollama pull` 여부를 확인한다" if want else "(비교할 서버 값 없음)")

## 3. 프롬프트와 호출 (파이프라인과 같은 방식)

`baseline` 은 `pipeline/risk.py` 의 `PROMPT` 원문이다. 새 안은 `PROMPTS` 에 이름을 붙여 더하고 `VARIANTS` 에 넣는다.
`v2_term_vs_tone` 은 Plan 03 표본에서 어긋난 유형(문장 분위기가 부정적일 뿐인데 '주의')을 겨냥한 **예시 안**이며 검증되지 않았다.

In [ ]:
import hashlib, re
from typing import Literal
import ollama
from pydantic import BaseModel, ValidationError

PROMPTS = {
    "baseline": """아래는 인터넷 표현 '{term}' 이 들어간 문장들이다. 사전의 뜻: {definition}
문장마다 세 가지를 붙여라.
on_topic: 그 문장이 '{term}' 을 **위 뜻의 유행어로** 쓰는가. 글자만 같은 딴 이야기(정책 이름·사람 이름·다른 뜻의 낱말)면 false.
polarity: 그 문장에서 '{term}' 이 쓰인 느낌 — positive(좋게·재미있게) | neutral | negative(비꼼·조롱·불쾌)
risk: 그 문장이 드러내는 위험 — hate(혐오·차별) | mockery(특정 대상 조롱) | gender(성별 갈등) | politics(정치) | sexual(선정성) | none
문장 번호(i)를 그대로 달아라. 판단이 어려우면 neutral, none.

문장:
{items}""",
    "v2_term_vs_tone": """아래는 인터넷 표현 '{term}' 이 들어간 문장들이다. 사전의 뜻: {definition}
문장마다 세 가지를 붙여라. 문장 전체의 기분이 아니라 **'{term}' 이라는 표현을 쓰는 방식**만 본다.
on_topic: 그 문장이 '{term}' 을 **위 뜻의 유행어로** 쓰는가. 글자만 같은 딴 이야기(정책 이름·사람 이름·다른 뜻의 낱말)면 false.
polarity: '{term}' 을 누군가를 비꼬거나 깎아내리려고 썼으면 negative, 재미있게·좋게 썼으면 positive, 그냥 썼으면 neutral.
  문장이 짜증·불평을 담고 있어도 '{term}' 자체를 그렇게 쓴 게 아니면 neutral 이다.
risk: '{term}' 이 그 문장에서 드러내는 위험 — hate(혐오·차별) | mockery(특정 대상 조롱) | gender(성별 갈등) | politics(정치) | sexual(선정성) | none
  문장 속 다른 낱말(욕설 등)이 거칠 뿐 '{term}' 과 상관없으면 none.
문장 번호(i)를 그대로 달아라. 판단이 어려우면 neutral, none.

문장:
{items}""",
}

# interpreters/local_chat.py · pipeline/local_llm.py 원문
ITEMS_RULE = "\n\nReturn JSON with a single key \"items\" holding the list."
JSON_RULE = "\n\n아래 모양대로 키 이름과 값 종류를 그대로 지켜 JSON 하나만 출력하라. 모든 키를 넣는다. null 은 모양에 |null 이 있는 키에만 쓰고, 그 외 키는 모르면 빈 문자열 \"\" 이나 빈 배열 [] 로 채운다. 설명이나 코드 펜스는 쓰지 않는다.\n모양: {schema}"
RETRY_RULE = "\n\n직전 응답은 위 모양과 맞지 않았다({error}). 같은 내용을 위 모양 그대로, 항목 수를 바꾸지 말고 다시 답하라."

RiskType = Literal["hate", "mockery", "gender", "politics", "sexual", "none"]


class EvidenceLabel(BaseModel):   # schemas.EvidenceLabel
    i: int
    polarity: Literal["positive", "neutral", "negative"] = "neutral"
    risk: RiskType = "none"
    on_topic: bool = True


class Items(BaseModel):
    items: list[EvidenceLabel]


def describe_schema(js: dict) -> str:
    defs = js.get("$defs", {})

    def render(node: dict) -> str:
        if "$ref" in node:
            node = defs[node["$ref"].rsplit("/", 1)[-1]]
        if "anyOf" in node:
            return "|".join(dict.fromkeys(render(n) for n in node["anyOf"]))
        if "enum" in node:
            return "|".join(json.dumps(v, ensure_ascii=False) for v in node["enum"])
        if "const" in node:
            return json.dumps(node["const"], ensure_ascii=False)
        t = node.get("type")
        if t == "object":
            return "{" + ", ".join(f'"{k}": {render(v)}' for k, v in node.get("properties", {}).items()) + "}"
        if t == "array":
            return "[" + render(node.get("items", {})) + ", ...]"
        return {"string": "str", "integer": "int", "number": "float", "boolean": "true|false", "null": "null"}.get(t, "any")

    return render(js)


SHAPE = describe_schema(Items.model_json_schema())


def close_brackets(t: str) -> str:
    stack, in_str, esc = [], False, False
    for ch in t:
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        elif ch == '"':
            in_str = True
        elif ch in "{[":
            stack.append("}" if ch == "{" else "]")
        elif ch in "}]" and stack:
            stack.pop()
    return t + ('"' if in_str else "") + "".join(reversed(stack))


def extract_json(text: str) -> str:
    t = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", text.strip())
    starts = [i for i in (t.find("{"), t.find("[")) if i >= 0]
    if not starts:
        raise ValueError(f"JSON 이 없음: {text[:80]!r}")
    i = min(starts)
    j = t.rfind("}" if t[i] == "{" else "]")
    cand = t[i: j + 1] if j >= i else t[i:]
    try:
        json.loads(cand)
        return cand
    except ValueError:
        fixed = close_brackets(t[i:].rstrip().rstrip("]}, \n\t"))
        json.loads(fixed)
        return fixed


def problem(e: Exception) -> str:
    if isinstance(e, ValidationError):
        return "; ".join(f"{'.'.join(map(str, x['loc'])) or '(전체)'}: {x['msg']}" for x in e.errors()[:4])[:300]
    return str(e).splitlines()[0][:200]


def valid_items(text: str):
    try:
        data = json.loads(extract_json(text))
        items = data.get("items") if isinstance(data, dict) else data
        if not isinstance(items, list):
            return None
    except ValueError:
        return None
    keep = []
    for it in items:
        try:
            keep.append(EvidenceLabel.model_validate(it))
        except ValidationError:
            continue
    return keep or None


client = ollama.Client(host="http://127.0.0.1:11434", timeout=180)


def chat(prompt: str) -> str:
    r = client.chat(model=MODEL, messages=[{"role": "user", "content": prompt}], think=False,
                    options={"temperature": TEMPERATURE, "num_ctx": NUM_CTX, "num_predict": MAX_TOKENS})
    return r.message.content


def structured(prompt: str):
    """(labels, attempts, status). local_chat.structured 와 같다: 틀리면 한 번 다시 묻고, 맞는 항목이 있으면 그것만 살린다."""
    ask = prompt + ITEMS_RULE + JSON_RULE.format(schema=SHAPE)
    error = ""
    for attempt in (1, 2):
        text = chat(ask + (RETRY_RULE.format(error=error) if error else ""))
        try:
            return Items.model_validate_json(extract_json(text)).items, attempt, "ok"
        except (ValidationError, ValueError) as e:
            error = problem(e)
            partial = valid_items(text)
            if partial is not None:
                return partial, attempt, "partial"
    return [], 2, f"failed: {error}"


def sha(t: str) -> str:
    return hashlib.sha256(t.encode()).hexdigest()


PROMPT_SHA = {name: sha(PROMPTS[name]) for name in PROMPTS}
SAME_CODE = {}
if SERVER_ENV:
    for key, mine in (("risk_prompt_sha256", PROMPT_SHA["baseline"]), ("json_rule_sha256", sha(JSON_RULE)),
                      ("retry_rule_sha256", sha(RETRY_RULE))):
        SAME_CODE[key] = SERVER_ENV.get(key) == mine
        print(key, "서버와 같음" if SAME_CODE[key] else "⚠ 서버와 다름 — 노트북의 원문을 서버 코드로 다시 맞춘다")
print(SHAPE)

## 4. 분류 실행

In [ ]:
import json, os, time
from pathlib import Path
from tqdm.auto import tqdm

signature_data = {"source_sha256": SOURCE_SHA256, "model": MODEL, "model_digest": MODEL_DIGEST,
                  "variants": {name: PROMPT_SHA[name] for name in VARIANTS},
                  "rules_sha256": sha(json.dumps([ITEMS_RULE, JSON_RULE, RETRY_RULE, SHAPE])),
                  "temperature": TEMPERATURE, "num_ctx": NUM_CTX,
                  "max_tokens": MAX_TOKENS, "per_term": PER_TERM, "terms": terms}
run_signature = sha(json.dumps(signature_data, sort_keys=True))
checkpoint_path = None
completed = {}
if RUN_DIR:
    run_path = Path(RUN_DIR)
    run_path.mkdir(parents=True, exist_ok=True)
    manifest_path = run_path / "signature.json"
    checkpoint_path = run_path / "checkpoint.json"
    if manifest_path.exists():
        if json.loads(manifest_path.read_text()) != signature_data:
            raise ValueError("RUN_DIR에 다른 입력 또는 모델 설정의 결과가 있습니다")
    else:
        if checkpoint_path.exists():
            raise ValueError("서명 없는 체크포인트는 재개할 수 없습니다")
        manifest_path.write_text(json.dumps(signature_data, ensure_ascii=False, indent=2))
    if checkpoint_path.exists():
        saved = json.loads(checkpoint_path.read_text())
        completed = {(item["call"]["variant"], item["call"]["term_id"]): item
                     for item in saved}

for variant in VARIANTS:
    for term_id in tqdm(terms, desc=variant):
        g = df[df.term_id == term_id].sort_values("i").head(PER_TERM)
        key = (variant, term_id)
        previous = completed.get(key)
        if previous and previous["call"]["status"] == "ok" and len(previous["labels"]) == len(g) and all(row["model_on_topic"] is not None for row in previous["labels"]):
            continue
        items = "\n".join(f"{int(r.i)}. {r.sentence[:220]}" for r in g.itertuples())
        prompt = PROMPTS[variant].format(term=g.term.iloc[0], definition=g.definition.iloc[0][:160], items=items)
        t0 = time.time()
        try:
            got, attempts, status = structured(prompt)
        except Exception as e:
            got, attempts, status = [], 0, f"error: {type(e).__name__}"
        call = {"variant": variant, "term_id": term_id, "seconds": round(time.time() - t0, 2),
                "attempts": attempts, "status": status, "sentences": len(g), "labeled": len(got)}
        by_i = {p.i: p for p in got}
        rows = []
        for r in g.itertuples():
            p = by_i.get(int(r.i))
            rows.append({"variant": variant, "term_id": term_id, "evidence_id": r.evidence_id, "i": int(r.i),
                         "model_on_topic": None if p is None else p.on_topic,
                         "model_polarity": None if p is None else (p.polarity if p.on_topic else "offtopic"),
                         "model_risk": None if p is None else (p.risk if p.on_topic else "none")})
        completed[key] = {"call": call, "labels": rows}
        if checkpoint_path:
            temporary = checkpoint_path.with_suffix(".tmp")
            temporary.write_text(json.dumps(list(completed.values()), ensure_ascii=False))
            os.replace(temporary, checkpoint_path)

ordered = [completed[(variant, term_id)] for variant in VARIANTS for term_id in terms]
labels = pd.DataFrame([row for item in ordered for row in item["labels"]])
calls = pd.DataFrame([item["call"] for item in ordered])
calls.groupby("variant").agg(calls=("term_id", "count"), p50_s=("seconds", "median"), max_s=("seconds", "max"),
                             retried=("attempts", lambda a: int((a > 1).sum())),
                             not_ok=("status", lambda s: int((s != "ok").sum())))

## 5. 용어 단위 판정 (규칙)

`pipeline/risk.judge` 의 문턱을 그대로 쓴다: 위험 근거 3건 이상이고 출처 두 곳 이상이면 사용 자제, 그 뜻으로 쓰인 문장이 5건 미만이면 판단 보류,
위험 근거 2건 이상이거나 부정 30% 이상이면 주의, 아니면 없음.
**빠진 것:** 표제어 자체의 욕설 목록(`match.unsafe`), 영상 해석의 기원 지적, 사전 뜻풀이의 주의점, 검수자 결정. 그래서 서버의 `term_risk_level` 과 다를 수 있다.

In [ ]:
AVOID_RISKY, CAUTION_RISKY, CAUTION_NEGATIVE, MIN_LABELED = 3, 2, 0.30, 5


def judge(rows: pd.DataFrame, pol: str, risk_col: str) -> str:
    rows = rows[rows[pol].notna() & (rows[pol] != "") & (rows[pol] != "offtopic")]
    risky = rows[rows[risk_col] != "none"]
    if len(risky) >= AVOID_RISKY and risky.source_domain.nunique() >= 2:
        return "avoid"
    if len(rows) < MIN_LABELED:
        return "undecided"
    if len(risky) >= CAUTION_RISKY or (rows[pol] == "negative").mean() >= CAUTION_NEGATIVE:
        return "caution"
    return "none"


joined = labels.merge(df, on=["term_id", "evidence_id", "i"], how="left")
term_rows = []
for (variant, term_id), g in joined.groupby(["variant", "term_id"]):
    term_rows.append({"variant": variant, "term_id": term_id, "term": g.term.iloc[0],
                      "model_level": judge(g, "model_polarity", "model_risk"),
                      "pipeline_labels_level": judge(g, "pipeline_polarity", "pipeline_risk") if (g.pipeline_polarity != "").any() else "",
                      "server_level": g.term_risk_human.iloc[0] or g.term_risk_level.iloc[0] or "undecided",
                      "human_level": g.human_level.iloc[0]})
term_df = pd.DataFrame(term_rows)
term_df.groupby("variant").model_level.value_counts().unstack(fill_value=0)

## 6. 비교

- **서버 분류와 문장 단위 일치** — 같은 문장에 서버(`pipeline_*`, 수집 서버의 12B)가 붙인 값과 비교한다.
- **사람 판정** — CSV 의 `human_*` 칸을 채웠으면 그것과도 비교한다(Plan 03 의 표본 20개 재검토에 쓴다).
- **용어 단위** — `pipeline_labels_level` 은 서버 분류로 이 노트북의 규칙을 돌린 값이라, 모델 차이만 본다. `server_level` 은 빠진 규칙까지 들어간 실제 수준이다.

In [ ]:
def rate(a: pd.Series, b: pd.Series) -> str:
    m = a.notna() & (a != "") & b.notna() & (b != "")
    return f"{(a[m] == b[m]).mean():.0%} ({int((a[m] == b[m]).sum())}/{int(m.sum())})" if m.any() else "—"


out = []
for variant, g in joined.groupby("variant"):
    srv_on = g.pipeline_polarity.map(lambda p: "" if p == "" else str(p != "offtopic"))
    hum_on = g.human_on_topic.str.lower().map(lambda v: {"1": "True", "true": "True", "0": "False", "false": "False"}.get(v, ""))
    mod_on = g.model_on_topic.map(lambda v: "" if v is None else str(v))
    t = term_df[term_df.variant == variant]
    out.append({"variant": variant,
                "on_topic vs 서버": rate(mod_on, srv_on), "polarity vs 서버": rate(g.model_polarity, g.pipeline_polarity),
                "risk vs 서버": rate(g.model_risk, g.pipeline_risk),
                "on_topic vs 사람": rate(mod_on, hum_on), "polarity vs 사람": rate(g.model_polarity, g.human_polarity),
                "risk vs 사람": rate(g.model_risk, g.human_risk),
                "수준 vs 서버 분류": rate(t.model_level, t.pipeline_labels_level),
                "수준 vs 사람": rate(t.model_level, t.human_level)})
pd.DataFrame(out).set_index("variant").T

In [ ]:
# 서버 분류와 수준이 다른 용어 — 프롬프트를 고칠 거리
diff = term_df[(term_df.pipeline_labels_level != "") & (term_df.model_level != term_df.pipeline_labels_level)]
diff[["variant", "term", "model_level", "pipeline_labels_level", "server_level", "human_level"]]

## 7. 결과 저장

In [ ]:
import datetime, json, os, platform, subprocess

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
outdir = RUN_DIR
os.makedirs(outdir, exist_ok=True)
labels.to_csv(f"{outdir}/sentence_labels.csv", index=False, encoding="utf-8-sig")
term_df.to_csv(f"{outdir}/term_levels.csv", index=False, encoding="utf-8-sig")
calls.to_csv(f"{outdir}/calls.csv", index=False, encoding="utf-8-sig")
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
meta = {"source": SOURCE, "synthetic": SYNTHETIC, "model": MODEL, "ollama": OLLAMA_VERSION, "gpu": gpu,
        "temperature": TEMPERATURE, "num_ctx": NUM_CTX, "max_tokens": MAX_TOKENS, "variants": VARIANTS,
        "model_digest": MODEL_DIGEST, "override": OVERRIDE, "server_env": SERVER_ENV, "same_code": SAME_CODE,
        "prompt_sha": {v: PROMPT_SHA[v] for v in VARIANTS}, "terms": len(terms), "python": platform.python_version(),
        "created": stamp, "source_sha256": SOURCE_SHA256, "run_signature": run_signature,
        "run_id": RUN_ID, "resumed_from": RESUME_FROM}
json.dump(meta, open(f"{outdir}/run.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(meta, ensure_ascii=False, indent=2))
print("결과:", outdir, "— Save Version 으로 돌렸으면 그 버전의 Output 에 남는다")